# terna_capacita_rinnovabile - notebook v0

Notebook v0 di validazione del mart in `dataset-incubator`.

- scopo: sanity check e lettura base del mart
- non sostituisce l'analisi pubblica
- evitare output pesanti o immagini embeddate nel commit


In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SLUG = "terna_capacita_rinnovabile"
MART_TABLE = "mart_regioni_fonti_nette"
METRICA = "potenza_totale_mw"
ANNO_TARGET = 2024

candidate_dir = Path.cwd()
if not (candidate_dir / "dataset.yml").exists():
    if (candidate_dir.parent / "dataset.yml").exists():
        candidate_dir = candidate_dir.parent
    else:
        raise FileNotFoundError(
            f"dataset.yml non trovato in {Path.cwd()} o nella cartella parent."
        )

mart_root = candidate_dir.parents[1] / "out" / "data" / "mart" / SLUG / str(ANNO_TARGET)
parquet_path = mart_root / f"{MART_TABLE}.parquet"
if not parquet_path.exists():
    raise FileNotFoundError(
        f"Mart non trovato per {SLUG}/{MART_TABLE} sotto {mart_root}. Eseguire prima toolkit run all."
    )

PARQUET_PATH = parquet_path
print(f"Candidate dir: {candidate_dir.name}")
print(f"Mart table: {PARQUET_PATH.name}")
print(f"Anno target: {ANNO_TARGET}")


Candidate dir: terna-capacita-rinnovabile
Mart table: mart_regioni_fonti_nette.parquet
Anno target: 2024


In [2]:
con = duckdb.connect()
df = con.execute("SELECT * FROM read_parquet(?)", [str(PARQUET_PATH)]).df()
print(f"Shape: {df.shape}")
display(df.dtypes)


Shape: (100, 5)


anno                   int32
regione                  str
fonti                    str
potenza_totale_mw    float64
record_count           int64
dtype: object

In [3]:
display(df.head(10))


,anno,regione,fonti,potenza_totale_mw,record_count
0,2024,Lombardia,Idrico,5155.33814,12
1,2024,Lombardia,Fotovoltaico,4959.29800,12
2,2024,Veneto,Fotovoltaico,3747.83800,7
3,2024,Puglia,Fotovoltaico,3626.85700,6
4,2024,Emilia-Romagna,Fotovoltaico,3574.10600,9
5,2024,Trentino-Alto Adige,Idrico,3563.05590,2
6,2024,Lazio,Fotovoltaico,3295.05000,5
7,2024,Puglia,Eolico,3228.22300,6
8,2024,Piemonte,Fotovoltaico,3083.35700,8
9,2024,Piemonte,Idrico,2787.24987,8


In [4]:
print("Null per colonna:")
display(df.isnull().sum())

negativi = int((df[METRICA] < 0).sum()) if pd.api.types.is_numeric_dtype(df[METRICA]) else "n/a"
print(
    f"\nRange {METRICA}: min={df[METRICA].min()}, max={df[METRICA].max()}, negativi={negativi}"
)


Null per colonna:


anno                 0
regione              0
fonti                0
potenza_totale_mw    2
record_count         0
dtype: int64


Range potenza_totale_mw: min=0.0, max=5155.338140000001, negativi=0


In [5]:
(
    df.groupby("regione")[METRICA]
    .sum()
    .sort_values(ascending=False)
    .plot(kind="bar", figsize=(12, 5), title=f"{METRICA} per regione ({ANNO_TARGET})")
)
plt.tight_layout()
plt.show()


## Note v0

- Slug: `terna_capacita_rinnovabile`
- Tabella mart: `mart_regioni_fonti_nette`
- Metrica guida: `potenza_totale_mw`
- Perimetro: dicembre 2023-2024; notebook puntato al mart 2024
- Questo notebook resta esplorativo e validativo in DI.
